# Reactions to countries: fine-tuning the model on our own labels

**Why.** The off-the-shelf model `yangheng/deberta-v3-base-absa` was trained on laptop and
restaurant reviews. On Reddit posts about the war it sees negativity where there is none: for it,
"Russia shelled Kyiv" is negative towards Ukraine. On the gold sample it got macro-F1 = 0.52 and
almost doubled the share of negative reactions. Here the model is fine-tuned on ~1600 labelled
pairs from our own dataset, while the gold sample (`gold_sample.csv`) stays a separate,
independent check.

**Before running**
1. *Runtime → Change runtime type → **T4 GPU***.
2. The **`reddit_reactions`** folder on Google Drive must contain:
   * `gold_sample.csv`: the labelled gold sample;
   * `train_sample.csv` and `train_sample2.csv`: **labelled** training samples
     (exactly these names, 1600 pairs in total);
   * `pairs_all.csv.gz`: all pairs (made by `build_all_pairs.py`), so every row of the dataset
     gets a reaction; without it only the sample `reaction_pairs.csv` is labelled.
3. *Runtime → Run all*. Training takes a few minutes, labelling all pairs a few hours
   (~1000 pairs/s). If Colab disconnects, just *Run all* again: the model is saved in
   `model_ft2/` and labelling resumes where it stopped.

**Safeguards.** If a `train_sample*.csv` is missing or unlabelled, the notebook stops at step 3
and says why. If the training data has changed, `model_ft2/` is retrained automatically. If
`reactions_all.csv` was started by a different model, the old file is moved to a backup and
labelling starts over.

**Output in the same folder:**
* `reactions_all.csv` (pair_id, reaction = 1 / 0 / -1): download it into `data/reactions/`
  and run `add_reactions.py`, which adds the reaction to every row of the dataset;
* `gold_predictions_ft2.csv`: model predictions on the gold sample, for error analysis.

Results of earlier runs (`model_ft/`, `reactions_ft.csv`) are not changed.

In [ ]:
# Step 1. Settings
from google.colab import drive
drive.mount('/content/drive')
!pip -q install sentencepiece protobuf

DRIVE_DIR  = "/content/drive/MyDrive/reddit_reactions"
PAIRS_CSV  = f"{DRIVE_DIR}/reaction_pairs.csv"
GOLD_CSV   = f"{DRIVE_DIR}/gold_sample.csv"
TRAIN_FILES = [f"{DRIVE_DIR}/train_sample.csv", f"{DRIVE_DIR}/train_sample2.csv"]
PAIRS_ALL  = f"{DRIVE_DIR}/pairs_all.csv.gz"     # all pairs (build_all_pairs.py)
MODEL_DIR  = f"{DRIVE_DIR}/model_ft2"           # fine-tuned model (~0.7 GB)
PREV_MODEL_DIR = f"{DRIVE_DIR}/model_ft"        # previous model, only for comparison on gold
ALL_OUT    = f"{DRIVE_DIR}/reactions_all.csv"   # reaction for every pair
OUT_CSV    = f"{DRIVE_DIR}/reactions_ft2.csv"   # without pairs_all, the sample is labelled instead
OLD_CSV    = f"{DRIVE_DIR}/reactions_ft.csv"    # previous model, for comparison
GOLD_PRED  = f"{DRIVE_DIR}/gold_predictions_ft2.csv"
BY_PERIOD  = f"{DRIVE_DIR}/reaction_by_period_ft2.csv"
BY_MONTH   = f"{DRIVE_DIR}/reaction_by_month_ft2.csv"
PLOT_PNG   = f"{DRIVE_DIR}/reaction_by_month_ft2.png"

BASE_MODEL = "yangheng/deberta-v3-base-absa-v1.1"
RETRAIN    = False        # True = retrain even on the same data
EPOCHS     = 4
LR         = 2e-5
TRAIN_BATCH = 16
VAL_SHARE  = 0.15         # validation share for picking the best epoch
SEED       = 42
BATCH_SIZE = 64           # set 32 on "CUDA out of memory"
MAX_LEN    = 256
SAVE_EVERY = 50_000       # how often to save to Drive (pairs)
USE_THREAD_WEIGHT = True  # weight 1 / sqrt(thread_n) in the summaries
MIN_PAIRS  = 30
PLOT_TARGETS = ["Ukraine", "Russia", "USA", "Germany", "Poland", "UK",
                "France", "Hungary", "Turkey", "China"]

In [ ]:
# Step 2. Helpers
import os, re, time, random, copy
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["negative", "neutral", "positive"]
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none")

LABEL_ALIASES = {
    "positive": "positive", "pos": "positive", "+": "positive", "1": "positive",
    "neutral": "neutral", "neu": "neutral", "0": "neutral",
    "negative": "negative", "neg": "negative", "-": "negative", "-1": "negative",
}


def norm_label(x):
    return LABEL_ALIASES.get(str(x).strip().lower())


def prep(text):
    text = re.sub(r"(?<!\w)@\w+", "@user", str(text))
    text = re.sub(r"https?://\S+", "http", text)
    return " ".join(text.split())


def load_labelled(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df["y"] = df["label"].map(norm_label)
    return df[df["y"].notna()].reset_index(drop=True)


def label_order(model):
    # model output indices in LABELS order (it differs between models)
    id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
    try:
        return [next(i for i, lab in id2label.items() if lab.startswith(x[:3])) for x in LABELS]
    except StopIteration:
        return [0, 1, 2]


def encode(tok, contexts, aspects):
    return tok([prep(c) for c in contexts], [str(a) for a in aspects],
               truncation="only_first", max_length=MAX_LEN, padding=True, return_tensors="pt")


@torch.no_grad()
def predict(model, tok, contexts, aspects, show=False):
    # probabilities [negative, neutral, positive]
    model.eval()
    order_out = label_order(model)
    contexts, aspects = list(contexts), list(aspects)
    out = np.zeros((len(contexts), 3), dtype=np.float32)
    order = np.argsort([len(str(c)) for c in contexts], kind="stable")
    steps = range(0, len(order), BATCH_SIZE)
    for s in (tqdm(steps, leave=False) if show else steps):
        idx = order[s:s + BATCH_SIZE]
        enc = encode(tok, [contexts[i] for i in idx], [aspects[i] for i in idx])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(**enc).logits
        out[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()[:, order_out]
    return out


def report(name, y_true, y_pred):
    f1 = f1_score(y_true, y_pred, average="macro", labels=LABELS)
    print(f"\n=== {name}: accuracy = {accuracy_score(y_true, y_pred):.3f} | macro-F1 = {f1:.3f}")
    print(classification_report(y_true, y_pred, labels=LABELS, digits=3, zero_division=0))
    print(pd.crosstab(pd.Series(y_true, name="gold"), pd.Series(y_pred, name="model")))
    true_share = pd.Series(y_true).value_counts(normalize=True).reindex(LABELS, fill_value=0)
    pred_share = pd.Series(y_pred).value_counts(normalize=True).reindex(LABELS, fill_value=0)
    print("Shares (gold -> model):",
          ", ".join(f"{l} {true_share[l]:.0%} -> {pred_share[l]:.0%}" for l in LABELS))
    return f1

In [ ]:
# Step 3. Training and gold data
parts, problems = [], []
for p in TRAIN_FILES:
    name = os.path.basename(p)
    if not os.path.exists(p):
        problems.append(f"{name} is missing")
        continue
    part = load_labelled(p)
    print(f"{'ok' if len(part) else 'EMPTY'} {name}: {len(part)} labelled pairs")
    if len(part) == 0:
        problems.append(f"{name} has an empty label column (unlabelled version of the file)")
    parts.append(part)
if problems:
    similar = sorted(f for f in os.listdir(DRIVE_DIR) if f.lower().startswith("train_sample"))
    raise FileNotFoundError(
        "Stopping so the model is not trained on incomplete data: " + "; ".join(problems)
        + f".\nFiles in {DRIVE_DIR}: {similar}"
        + "\nPut the labelled file there under exactly this name (no '(1)' etc.) and Run all again.")
train_all = pd.concat(parts, ignore_index=True).drop_duplicates("pair_id")
gold = load_labelled(GOLD_CSV)
# gold is only for evaluation, so the same pairs and texts are removed from training
overlap = train_all["pair_id"].isin(gold["pair_id"]) | train_all["context"].isin(gold["context"])
train_all = train_all[~overlap].reset_index(drop=True)
print(f"Training pairs: {len(train_all)} | gold: {len(gold)} | overlaps removed: {int(overlap.sum())}")
print("Train:", train_all["y"].value_counts().to_dict())
print("Gold: ", gold["y"].value_counts().to_dict())
if len(train_all) < 200:
    print("Warning: too few labelled pairs in train_sample*.csv, check the label column")

In [ ]:
# Step 4. Fine-tuning (or loading the already fine-tuned model)
from sklearn.model_selection import train_test_split
from transformers import get_linear_schedule_with_warmup


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(seed)


import hashlib, json


def train_fingerprint():
    # fingerprint of data and settings; if it changes, the model has to be retrained
    rows = sorted(zip(train_all["pair_id"], train_all["y"]))
    key = json.dumps([rows, BASE_MODEL, EPOCHS, LR, TRAIN_BATCH, VAL_SHARE, SEED, MAX_LEN])
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:12]


FP = train_fingerprint()
INFO_JSON = f"{MODEL_DIR}/train_info.json"
saved_fp = None
if os.path.exists(INFO_JSON):
    with open(INFO_JSON, encoding="utf-8") as fh:
        saved_fp = json.load(fh).get("fingerprint")
need_train = RETRAIN or not os.path.isdir(MODEL_DIR) or saved_fp != FP
if os.path.isdir(MODEL_DIR) and saved_fp != FP and not RETRAIN:
    print("model_ft2/ was trained on different (or unknown) data, retraining")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(DEVICE)
base_pred = np.array(LABELS)[predict(base, tok, gold["context"], gold["aspect"]).argmax(1)]
f1_base = report("base model (before fine-tuning)", gold["y"], base_pred)

if not need_train:
    print(f"\nLoading fine-tuned model from {MODEL_DIR} (trained on the same {len(train_all)} pairs)")
    del base
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
else:
    set_seed(SEED)
    model = base
    order_out = label_order(model)                       # LABELS -> model output index
    to_id = {lab: order_out[i] for i, lab in enumerate(LABELS)}
    tr, va = train_test_split(train_all, test_size=VAL_SHARE, stratify=train_all["y"],
                              random_state=SEED)
    counts = tr["y"].value_counts()
    w = torch.zeros(len(LABELS))
    for lab in LABELS:                                   # rare classes get more weight
        w[to_id[lab]] = (len(tr) / (3 * max(counts.get(lab, 1), 1))) ** 0.5
    loss_fn = torch.nn.CrossEntropyLoss(weight=w.to(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps_per_epoch = int(np.ceil(len(tr) / TRAIN_BATCH))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps_per_epoch * EPOCHS),
                                            steps_per_epoch * EPOCHS)
    best_f1, best_state = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.random.permutation(len(tr))
        losses = []
        for s in tqdm(range(0, len(tr), TRAIN_BATCH), desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            b = tr.iloc[perm[s:s + TRAIN_BATCH]]
            enc = {k: v.to(DEVICE) for k, v in encode(tok, b["context"], b["aspect"]).items()}
            target = torch.tensor([to_id[y] for y in b["y"]], device=DEVICE)
            loss = loss_fn(model(**enc).logits.float(), target)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            losses.append(loss.item())
        va_pred = np.array(LABELS)[predict(model, tok, va["context"], va["aspect"]).argmax(1)]
        f1_va = f1_score(va["y"], va_pred, average="macro", labels=LABELS)
        print(f"Epoch {epoch}: loss = {np.mean(losses):.3f} | validation macro-F1 = {f1_va:.3f}")
        if f1_va > best_f1:
            best_f1 = f1_va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.save_pretrained(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    with open(INFO_JSON, "w", encoding="utf-8") as fh:
        json.dump({"fingerprint": FP, "n_train": len(train_all), "best_val_f1": round(best_f1, 4),
                   "files": [os.path.basename(p) for p in TRAIN_FILES]}, fh, ensure_ascii=False)
    print(f"Best epoch: validation macro-F1 = {best_f1:.3f}, saved to {MODEL_DIR}")

probs_gold = predict(model, tok, gold["context"], gold["aspect"])
gold["pred_base"] = base_pred
gold["pred_ft2"] = np.array(LABELS)[probs_gold.argmax(1)]
f1_ft = report("fine-tuned model (model_ft2)", gold["y"], gold["pred_ft2"])
scores = {"base model": f1_base, "model_ft2 (new)": f1_ft}
MODEL_TAG = f"model_ft2:{FP}"           # which model labelled the pairs, so results don't get mixed
if os.path.isdir(PREV_MODEL_DIR):
    prev = AutoModelForSequenceClassification.from_pretrained(PREV_MODEL_DIR).to(DEVICE)
    gold["pred_ft1"] = np.array(LABELS)[predict(prev, tok, gold["context"], gold["aspect"]).argmax(1)]
    scores["model_ft (previous)"] = f1_score(gold["y"], gold["pred_ft1"], average="macro",
                                             labels=LABELS)
    if scores["model_ft (previous)"] > f1_ft:
        model = prev                     # label with the better of the two
        MODEL_TAG = "model_ft"
        print("\nThe previous model is better on gold, using it for labelling (model_ft)")
    else:
        del prev
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
gold.to_csv(GOLD_PRED, index=False, encoding="utf-8-sig")
print("\nmacro-F1 on gold:", " | ".join(f"{k}: {v:.3f}" for k, v in scores.items()))

In [ ]:
# Step 5. Labelling pairs (resumes where it stopped)
FULL = os.path.exists(PAIRS_ALL)
CODE = np.array([-1, 0, 1])             # negative / neutral / positive
t0 = time.time()


def check_same_model(out_path):
    # resuming only works with the same model, otherwise the old file goes to a backup
    tag_path = out_path + ".model.txt"
    old_tag = open(tag_path, encoding="utf-8").read().strip() if os.path.exists(tag_path) else None
    if os.path.exists(out_path) and old_tag != MODEL_TAG:
        backup = out_path.replace(".csv", time.strftime("_old_%Y%m%d_%H%M.csv"))
        os.rename(out_path, backup)
        print(f"{os.path.basename(out_path)} was labelled by another model, moved to "
              f"{os.path.basename(backup)}, starting over")
    with open(tag_path, "w", encoding="utf-8") as fh:
        fh.write(MODEL_TAG)


check_same_model(ALL_OUT if FULL else OUT_CSV)
print("Labelling with:", MODEL_TAG)
if FULL:
    # read all pairs in chunks so millions of rows don't sit in memory
    done = set()
    if os.path.exists(ALL_OUT):
        done = set(pd.read_csv(ALL_OUT, dtype=str, usecols=["pair_id"])["pair_id"])
    print(f"Counting pairs in {os.path.basename(PAIRS_ALL)}...")
    total = sum(len(c) for c in pd.read_csv(PAIRS_ALL, usecols=["pair_id"], chunksize=500_000))
    print(f"Total pairs: {total:,} | done: {len(done):,} | left: {total - len(done):,}")
    bar = tqdm(total=total, initial=len(done), desc="Pairs", unit=" pairs")
    seen = 0
    for part in pd.read_csv(PAIRS_ALL, dtype=str, keep_default_na=False, chunksize=SAVE_EVERY):
        seen += len(part)
        if seen <= len(done) and part["pair_id"].iloc[-1] in done:
            continue                     # chunk already labelled
        part = part[~part["pair_id"].isin(done)]
        if len(part):
            probs = predict(model, tok, part["context"], part["aspect"])
            pd.DataFrame({"pair_id": part["pair_id"].values,
                          "reaction": CODE[probs.argmax(1)]}).to_csv(
                ALL_OUT, mode="a", header=not os.path.exists(ALL_OUT), index=False)
            bar.update(len(part))
    bar.close()
    print(f"Done in {time.strftime('%H:%M:%S', time.gmtime(time.time() - t0))}: {ALL_OUT}")
    print("Download reactions_all.csv into data/reactions/ and run add_reactions.py")
else:
    pairs = pd.read_csv(PAIRS_CSV, dtype=str, keep_default_na=False,
                        usecols=["pair_id", "aspect", "context"])
    done = set()
    if os.path.exists(OUT_CSV):
        done = set(pd.read_csv(OUT_CSV, dtype=str, usecols=["pair_id"])["pair_id"])
    todo = pairs[~pairs["pair_id"].isin(done)].reset_index(drop=True)
    print(f"No pairs_all.csv.gz, labelling the sample. Total pairs: {len(pairs):,} | "
          f"done: {len(done):,} | left: {len(todo):,}")
    for start in tqdm(range(0, len(todo), SAVE_EVERY), desc="Parts"):
        part = todo.iloc[start:start + SAVE_EVERY]
        probs = predict(model, tok, part["context"], part["aspect"], show=True)
        pd.DataFrame({
            "pair_id": part["pair_id"].values,
            "label": np.array(LABELS)[probs.argmax(1)],
            "p_neg": probs[:, 0].round(4), "p_neu": probs[:, 1].round(4),
            "p_pos": probs[:, 2].round(4), "model": "absa_ft2",
        }).to_csv(OUT_CSV, mode="a", header=not os.path.exists(OUT_CSV), index=False)
    print(f"Done in {time.strftime('%H:%M:%S', time.gmtime(time.time() - t0))}: {OUT_CSV}")

In [ ]:
# Step 6. Summaries (sample only; for all pairs add_reactions.py makes them)
if FULL:
    print("All pairs are labelled. Monthly and period summaries come from add_reactions.py")
else:
    cols = ["pair_id", "target", "period", "month", "weight", "thread_n"]
    meta = pd.read_csv(PAIRS_CSV, dtype=str, keep_default_na=False, usecols=cols)
    meta["sw"] = pd.to_numeric(meta["weight"], errors="coerce").fillna(1.0)   # sample weight
    meta["w"] = meta["sw"]
    if USE_THREAD_WEIGHT:
        tn = pd.to_numeric(meta["thread_n"], errors="coerce").fillna(1).clip(lower=1)
        meta["w"] = meta["w"] / np.sqrt(tn)


    def summarize(res_path, by):
        res = pd.read_csv(res_path, dtype={"pair_id": str}).drop_duplicates("pair_id", keep="last")
        d = meta.merge(res[["pair_id", "label"]], on="pair_id", how="inner")
        for lab in LABELS:
            d[lab] = (d["label"] == lab).astype(float) * d["w"]
        g = d.groupby(by)
        s = g[LABELS].sum().div(g["w"].sum(), axis=0)
        s.columns = [f"share_{c}" for c in LABELS]
        s["net"] = s["share_positive"] - s["share_negative"]   # from -1 to +1
        s["n_pairs"] = g.size()
        s["weighted_pairs"] = g["sw"].sum().round(0)
        return s[s["n_pairs"] >= MIN_PAIRS].round(3).reset_index()


    by_period = summarize(OUT_CSV, ["target", "period"])
    by_month = summarize(OUT_CSV, ["target", "month"])
    by_period.to_csv(BY_PERIOD, index=False)
    by_month.to_csv(BY_MONTH, index=False)

    net = by_period[by_period["target"].isin(PLOT_TARGETS)].pivot(index="target", columns="period",
                                                                  values="net")
    if os.path.exists(OLD_CSV):
        old = summarize(OLD_CSV, ["target", "period"])
        old = old[old["target"].isin(PLOT_TARGETS)].pivot(index="target", columns="period", values="net")
        net = pd.concat({"new model": net, "old model": old}, axis=1)
    print("Reaction balance (net = share positive - share negative):")
    print(net.round(2))

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for ax, year in zip(axes, ["2019", "2022"]):
        part = by_month[by_month["month"].str.startswith(year) & by_month["target"].isin(PLOT_TARGETS)]
        for target, grp in part.groupby("target"):
            ax.plot(grp["month"], grp["net"], marker="o", label=target)
        ax.axhline(0, color="grey", lw=0.8)
        ax.set_title(f"{year}: reaction balance (model_ft2)")
        ax.tick_params(axis="x", rotation=45)
    axes[0].set_ylabel("net")
    axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(PLOT_PNG, dpi=150)
    plt.show()
    print(f"Saved: {BY_PERIOD}\nSaved: {BY_MONTH}\nPlot: {PLOT_PNG}")